In [ ]:
# Import numpy for arrays and matplotlib for drawing the numbers
import numpy as np
import matplotlib.pyplot as plt
# import scipy.special for the sigmoid function expit()
import scipy.special 


In [ ]:

# draw number

def draw_number(image_num):
    # Open the 100 training samples in read mode
    # data_file = open('mnist_train_100.csv','r')
    # data_file = open('mnist_train.csv','r')
    # data_file = open('fashion_mnist_train_100.csv','r')
    # data_file = open('fashion_mnist_train.csv','r')
    # data_file = open('mnist_test_100.csv','r')
    # data_file = open('mnist_test.csv','r')
    # data_file = open('fashion_mnist_test_100.csv','r')
    data_file = open('fashion_mnist_test.csv','r')

    # Read all of the lines from the file into memory
    data_list = data_file.readlines()

    # Close the file (we are done with it)
    data_file.close()

    # Take the first line (data_list index 0, the first sample), and split it up based on the commas
    # all_values now contains a list of [label, pixel 1, pixel 2, pixel 3, ... , pixel 784]
    all_values = data_list[image_num].split(',')

    # Take the long list of pixels (but not the label), and reshape them to a 2D array of pixels
    image_array = np.asfarray(all_values[1:]).reshape((28, 28))

    # PLot this 2D array as an image, use the grey colour map and don't interpolate
    plt.imshow(image_array, cmap='Greys', interpolation='None')
    plt.show()

In [ ]:
# Neural network class definition
class NeuralNetwork:
    # Init the network, this gets run whenever we make a new instance of this class
    def __init__(self, input_nodes, hidden_nodes, output_nodes, learning_rate):
        # Set the number of nodes in each input, hidden and output layer
        self.i_nodes = input_nodes
        self.h_nodes = hidden_nodes
        self.o_nodes = output_nodes

        # Weight matrices, with (input -> hidden) and who (hidden -> output)
        self.wih = np.random.normal(0.0, pow(self.h_nodes, -0.5), (self.h_nodes, self.i_nodes))
        self.who = np.random.normal(0.0, pow(self.o_nodes, -0.5), (self.o_nodes, self.h_nodes))

        # Set the learning rate
        self.lr = learning_rate

        # Set the activation function, the logistic sigmoid
        self.activation_function = lambda x: scipy.special.expit(x)
        # self.activation_function = lambda x: np.asarray([max(0, i) for i in x])

    # Train the network using back-propagation of errors
    def train(self, inputs_list, targets_list):
        # Convert inputs into 2D arrays
        inputs_array = np.array(inputs_list, ndmin=2).T
        targets_array = np.array(targets_list, ndmin=2).T

        # Calculate signals into hidden layer
        hidden_inputs = np.dot(self.wih, inputs_array)

        # Calculate signals emerging from hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)

        # Calculate signals into final output layer
        final_inputs = np.dot(self.who, hidden_outputs)

        # Calculate the signals emerging from final output layer
        final_outputs = self.activation_function(final_inputs)

        # Current error is (target - actual)
        output_errors = targets_array - final_outputs

        # Hidden layer errors are the output errors, split by the weights, recombined at hidden nodes
        hidden_errors = np.dot(self.who.T, output_errors)

        # Update the weights for the links between the hidden and output layers
        self.who += self.lr*np.dot((output_errors*final_outputs*(1.0 - final_outputs)), np.transpose(hidden_outputs))
        # Update the weights for the links between the input and hidden layers
        self.wih += self.lr*np.dot((hidden_errors*hidden_outputs*(1.0 - hidden_outputs)), np.transpose(inputs_array))

    # Query the network
    def query(self, inputs_list):
        # Converts the inputs list into a 2D array
        inputs_array = np.array(inputs_list, ndmin=2).T

        # Calcualte the signals into hidden layer
        hidden_inputs = np.dot(self.wih, inputs_array) 

        # Calculate output from the hidden layer
        hidden_outputs = self.activation_function(hidden_inputs)

        # Calculate signals into final layer
        final_inputs = np.dot(self.who, hidden_outputs)

        # Calculate outputs from the final layer
        final_outputs = self.activation_function(final_inputs)
        # print(final_outputs)
        return final_outputs


In [ ]:
# parameters

# Exercise 1
# input_nodes = 784
# hidden_nodes = 100
# output_nodes = 10
# learning_rate = 0.3

# Exercise 4
input_nodes = 784
hidden_nodes = 900 #100
output_nodes = 10 
learning_rate = 0.1 #0.3

n = NeuralNetwork(input_nodes, hidden_nodes, output_nodes, learning_rate)

In [ ]:
# training

# Load the MNIST 100 training samples CSV file into a list
# training_data_file = open("mnist_train_100.csv",'r')
# training_data_file = open("mnist_train.csv",'r')
# training_data_file = open("fashion_mnist_train_100.csv",'r')
training_data_file = open("fashion_mnist_train.csv",'r')
training_data_list = training_data_file.readlines()
training_data_file.close()

# Train the neural network on each training sample
epochs = 1
for e in range(epochs):
#go through all datasets
    i = 1
    for record in training_data_list:
        # Split the record by the commas
        all_values = record.split(',')
        # Scale and shift the inputs from 0..255 to 0.01..1
        inputs = (np.asfarray(all_values[1:]) / 255.0*0.99) + 0.01
        #create the target output values (all 0.01, except the desired label which is 0.99)
        targets = np.zeros(output_nodes) + 0.01
        # All_values[0] is the target label for this record
        targets[int(all_values[0])] = 0.99
        # Train the network
        n.train(inputs, targets)
        # print("epoch: ", e, " : ", i, " out of: 60.000")
        i = i + 1
        pass
    print(e)
    pass
print("done")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
done


In [ ]:
# testing

# Load the MNIST test samples CSV file into a list
# test_data_file = open("mnist_test_10.csv",'r')
# test_data_file = open("mnist_test.csv",'r')
# test_data_file = open("fashion_mnist_test_10.csv",'r')
test_data_file = open("fashion_mnist_test.csv",'r')
test_data_list = test_data_file.readlines()
test_data_file.close()

# Scorecard list for how well the network performs, initially empty
scorecard = []

# Loop through all of the records in the test data set
for record in test_data_list:
    # Split the record by the commas
    all_values = record.split(',')
    # The correct label is the first value
    correct_label = int(all_values[0])
    print(correct_label, "Correct label")
    # Scale and shift the inputs
    inputs = (np.asfarray(all_values[1:])/255.0*0.99)+0.01
    # Query the network
    outputs = n.query(inputs)
    # The index of the highest value output corresponds to the label
    label = np.argmax(outputs)
    print(label,"Network label")
    # Append either a 1 or a 0 to the scorecard list
    if (label == correct_label):
        scorecard.append(1)
    else:
        scorecard.append(0)
        pass
    pass

# Calculate the performance score, the fraction of correct answers
scorecard_array = np.asarray(scorecard)
# Exercise 2
print("Performance = ", (scorecard_array.sum()/scorecard_array.size)*100, '%')

9 Correct label
9 Network label
2 Correct label
2 Network label
1 Correct label
1 Network label
1 Correct label
1 Network label
6 Correct label
6 Network label
1 Correct label
1 Network label
4 Correct label
4 Network label
6 Correct label
6 Network label
5 Correct label
5 Network label
7 Correct label
7 Network label
4 Correct label
4 Network label
5 Correct label
5 Network label
7 Correct label
5 Network label
3 Correct label
3 Network label
4 Correct label
4 Network label
1 Correct label
1 Network label
2 Correct label
2 Network label
4 Correct label
2 Network label
8 Correct label
8 Network label
0 Correct label
0 Network label
2 Correct label
2 Network label
5 Correct label
7 Network label
7 Correct label
7 Network label
9 Correct label
7 Network label
1 Correct label
1 Network label
4 Correct label
2 Network label
6 Correct label
6 Network label
0 Correct label
0 Network label
9 Correct label
9 Network label
3 Correct label
3 Network label
8 Correct label
8 Network label
8 Correc

In [ ]:
# Exercise 3: Display images (I think I could have got these classifications...)
# for i in range(len(scorecard_array)):
#     if scorecard_array[i] == 0:
#         draw_number(i)
